# **Calidad del Aire en Colombia: Análisis y Predicción** 
**Por:** Juan Esteban Hernandez Gualtero

## **Fase 1: Tema, Justificación y Preguntas Clave**

### **El Tema: Análisis Territorial y Temporal de la Calidad del Aire en Colombia.**
El dataset contiene registros de variables críticas (como material particulado $PM_{10}$, $PM_{2.5}$, dirección del viento, etc.) monitoreadas por distintas autoridades ambientales en múltiples municipios del país.

### **Justificación:**
La contaminación atmosférica es uno de los principales riesgos ambientales para la salud pública en Colombia. Analizar este dataset no solo permite mapear los focos críticos de contaminación, sino también evaluar la confiabilidad de las mediciones (representatividad temporal) y el comportamiento de los contaminantes frente a los límites legales actuales. Esto es fundamental antes de entrenar cualquier modelo predictivo, ya que el comportamiento geográfico y la naturaleza de cada contaminante dictarán la estrategia de modelado.

### **Preguntas Clave:**
- Distribución de Contaminantes: ¿Cuáles son las variables más medidas en el país y qué regiones (Departamentos/Municipios) presentan los promedios anuales más altos de contaminantes críticos como $PM_{2.5}$ y $PM_{10}$?
- Cumplimiento Normativo: ¿Qué porcentaje de las estaciones reportan excedencias de los límites legales (Excedencias limite actual) y en qué tipo de estaciones (Fijas vs. Móviles) es más frecuente este fenómeno?
- Comportamiento de Extremos: ¿Qué relación existe entre la mediana, el percentil 98 y los valores máximos registrados para los principales contaminantes?

## **Fase 2: Adquisición y Comprensión de Datos**

### **Fuente de los Datos:**
https://www.datos.gov.co/Ambiente-y-Desarrollo-Sostenible/Calidad-del-Aire-en-Colombia-Promedio-Anual-/kekd-7v7h

### **Carga y primera inspección del dataset:**
En este punto, se carga el dataset para planear un panorama general, y tener una primera visualización de la información con la que se estará trabajando a lo largo de las fases.

In [30]:
import pandas as pd
import numpy as np

df = pd.read_csv('../Calidad_del_Aire_en_Colombia_(Promedio_Anual)_20260622.csv')

# 1. Inspeccion de la forma
print("--- Dimensiones del Dataset ---")
print(f"Filas: {df.shape[0]}, Columnas: {df.shape[1]}")
print("\n")

# 2. Visualizacion rapida
print("--- Primeras 5 filas ---")
display(df.head())

# 3. Info general: tipos de datos y valores no nulos
print("\n--- Información General y Tipos de Datos ---")
print(df.info())

--- Dimensiones del Dataset ---
Filas: 29683, Columnas: 28


--- Primeras 5 filas ---


,ID Estacion,Autoridad Ambiental,Estación,Latitud,Longitud,Variable,Unidades,Tiempo de exposición (horas),Año,Promedio,...,Fechas/horas del máximo,Mínimo,Fechas/horas del mínimo,Días de excedencias,Código del Departamento,Nombre del Departamento,Código del Municipio,Nombre del Municipio,Tipo de Estación,Ubicacion
0,"9,020",AMVA,I.E. COL. COLOMBIA,6.378517,-75.443986,DViento,deg,1,"2,011",256.8,...,29/11/2011 1:00,1.3,29/11/2011 7:00,0,5,ANTIOQUIA,5308.0,GIRARDOTA,Fija,POINT (-75.443986 6.378517)
1,"9,020",AMVA,I.E. COL. COLOMBIA,6.378517,-75.443986,DViento,deg,24,"2,011",257.4,...,16/11/2011 0:00,99.7,8/11/2011 0:00,0,5,ANTIOQUIA,5308.0,GIRARDOTA,Fija,POINT (-75.443986 6.378517)
2,"9,020",AMVA,I.E. COL. COLOMBIA,6.378517,-75.443986,PLiquida,mm,1,"2,011",4,...,20/12/2011 2:00,1.5,07/11/2011 23:00:00 - 08/11/2011 05:00:00 - 13...,0,5,ANTIOQUIA,5308.0,GIRARDOTA,Fija,POINT (-75.443986 6.378517)
3,"9,020",AMVA,I.E. COL. COLOMBIA,6.378517,-75.443986,P,mmHg,1,"2,011",645.9,...,12/09/2011 10:00,641.6,27/10/2011 17:00,0,5,ANTIOQUIA,5308.0,GIRARDOTA,Fija,POINT (-75.443986 6.378517)
4,"9,020",AMVA,I.E. COL. COLOMBIA,6.378517,-75.443986,P,mmHg,24,"2,011",645.9,...,20/10/2011 0:00,644,27/10/2011 0:00,0,5,ANTIOQUIA,5308.0,GIRARDOTA,Fija,POINT (-75.443986 6.378517)



--- Información General y Tipos de Datos ---
<class 'pandas.DataFrame'>
RangeIndex: 29683 entries, 0 to 29682
Data columns (total 28 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   ID Estacion                           29683 non-null  str    
 1   Autoridad Ambiental                   29683 non-null  str    
 2   Estación                              29683 non-null  str    
 3   Latitud                               29683 non-null  float64
 4   Longitud                              29683 non-null  float64
 5   Variable                              29683 non-null  str    
 6   Unidades                              29683 non-null  str    
 7   Tiempo de exposición (horas)          29683 non-null  int64  
 8   Año                                   29683 non-null  str    
 9   Promedio                              29683 non-null  str    
 10  Suma                                  29683 non

### **Primeros vistazos:**
Se dispone de un dataset con numerosos registros (mas de 10k). Hay en total 9 columnas numéricas (int y float), y 19 de texto (string). A simple vista parece ser un dataset con pocos valores nulos entre sus registros a pesar de su tamaño, lo que podria facilitar el desarrollo de las fases posteriores.


## **Fase 3: Diagnóstico de Calidad de Datos**

In [31]:
print("==========================================================")
print("                    EJECUTANDO DIAGNOSTICO                ")
print("==========================================================\n")

# 1. Identificar columnas numericas catalogadas como string
columnas_numericas_atrapadas = [
    'Promedio', 'Suma', 'No. de datos', 'Mediana', 
    'Percentil 98', 'Máximo', 'Mínimo', 'ID Estacion'
]

print("1. ANALISIS DE CARACTERES ESPECIALES EN COLUMNAS NUMERICAS:")
print("-" * 60)
for col in columnas_numericas_atrapadas:
    if col in df.columns:
        # Verificar si la columna se cargo como objeto/string
        es_objeto = df[col].dtype == 'object'
        
        # Contar cuantos registros contienen comas (separadores de miles propensos a romper Pandas)
        con_coma = df[col].astype(str).str.contains(',').sum()
        
        # Contar cuantos contienen texto o letras (ej: "ND", "Sin dato")
        con_texto = df[col].astype(str).str.contains('[a-zA-Z]', regex=True).sum()
        
        print(f"Columna: '{col}' (Tipo original: {df[col].dtype})")
        print(f"  -> Registros con comas: {con_coma} ({con_coma/len(df)*100:.2f}%)")
        print(f"  -> Registros con texto/letras: {con_texto} ({con_texto/len(df)*100:.2f}%)")
        print("-" * 60)

# 2. Busqueda de nulos camuflados (espacios en blanco)
print("\n2. DETECCION DE ESPACIOS EN BLANCO:")
print("-" * 60)
for col in df.columns:
    if df[col].dtype == 'object':
        espacios_blanco = df[col].astype(str).str.strip().eq('').sum()
        if espacios_blanco > 0:
            print(f"  - Columna '{col}': {espacios_blanco} registros son solo espacios vacios.")


# 3. Revision de consistencia en registros geograficos e ID
print("\n3. CONSISTENCIA GEOGRAFICA Y CATEGORICA:")
print("-" * 60)
nulos_municipio_id = df['Código del Municipio'].isnull().sum()
nulos_municipio_nom = df['Nombre del Municipio'].isnull().sum()
print(f"  - Registros sin Código de Municipio: {nulos_municipio_id}")
print(f"  - Registros sin Nombre de Municipio: {nulos_municipio_nom}")
print(f"  - Años unicos registrados en el dataset: {df['Año'].unique()}")
print(f"  - Cantidad de contaminantes/variables distintas: {df['Variable'].nunique()}")

                    EJECUTANDO DIAGNOSTICO                

1. ANALISIS DE CARACTERES ESPECIALES EN COLUMNAS NUMERICAS:
------------------------------------------------------------
Columna: 'Promedio' (Tipo original: str)
  -> Registros con comas: 369 (1.24%)
  -> Registros con texto/letras: 0 (0.00%)
------------------------------------------------------------
Columna: 'Suma' (Tipo original: str)
  -> Registros con comas: 25074 (84.47%)
  -> Registros con texto/letras: 0 (0.00%)
------------------------------------------------------------
Columna: 'No. de datos' (Tipo original: str)
  -> Registros con comas: 13217 (44.53%)
  -> Registros con texto/letras: 0 (0.00%)
------------------------------------------------------------
Columna: 'Mediana' (Tipo original: str)
  -> Registros con comas: 322 (1.08%)
  -> Registros con texto/letras: 0 (0.00%)
------------------------------------------------------------
Columna: 'Percentil 98' (Tipo original: str)
  -> Registros con comas: 1169 (3.94%

Tras inspeccionar la estructura general del dataset, se han identificado anomalías críticas que impiden el análisis estadístico directo y el entrenamiento de modelos de Machine Learning. A continuación, se detallan los hallazgos:

### **1. Variables Cuantitativas en Formato de texto**
Las variables esenciales para el estudio de la calidad del aire (Promedio, Suma, Mediana, Percentil 98, Máximo y Mínimo) fueron interpretadas por Pandas como tipo object (texto) en lugar de float64, aunque no se encontraron registros con datos alfanuméricos dentro de las columnas analizadas.

- Causa principal: El uso de comas (,) como separadores de miles en el archivo de origen (por ejemplo, valores como "321,637.50" o "4,418.76"). Al no reconocer la coma como un formato puramente numérico estándar, Python procesa toda la columna como una cadena de caracteres.

- Impacto: Imposibilidad de calcular distribuciones, correlaciones o matrices de covarianza de forma nativa.

### **2. Nulos Camuflados**
Además de las comas, existen registros en las columnas numéricas que contienen caracteres alfabéticos directos (letras). Esto suele corresponder a convenciones institucionales para reportar datos faltantes (como "ND", "S.D." o "No Registra"). Igualmente, espacios en blanco suelen afectar el análisis, al parecer que si se ingresaron los datos, cuando solo hay un espacio vacío (" "). Los anteriores casos actúan como "nulos ocultos" que distorsionan el conteo inicial de df.isnull().sum(). Por fortuna, no se encontraron registros que dieran con la presencia de algunos de estos problemas.

### **3. Inconcistencia Geográfica**
Existe una discrepancia menor en la completitud de la ubicación: la columna Código del Municipio presenta 4 valores nulos absolutos, mientras que Nombre del Municipio está 100% diligenciado. Esto evidencia un problema de consistencia en el cruce de diccionarios de datos geográficos de origen.



## **4. Limpieza de Datos**

In [32]:
# Crear una copia para trabajar la limpieza
df_clean = df.copy()

print("==========================================================")
print("                      LIMPIEZA DE DATOS                    ")
print("==========================================================\n")

# 1. Definir columnas que deben ser puramente numericas
columnas_a_convertir = [
    'Promedio', 'Suma', 'No. de datos', 'Mediana', 
    'Percentil 98', 'Máximo', 'Mínimo', 'Año', 'ID Estacion'
]

# 2. Proceso de limpieza y casteo
for col in columnas_a_convertir:
    if col in df_clean.columns:
        # Convertir a string, limpiar espacios en blanco y eliminar comas
        df_clean[col] = df_clean[col].astype(str).str.strip().str.replace(',', '', regex=False)
        
        # Forzar a numerico. Espacios vacios o textos se vuelven NaN automaticamente
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# 3. Ajustar tipos de datos para enteros que aceptan nulos (evita que se vuelvan floats con .0)
enteros_con_nan = ['Año', 'ID Estacion', 'No. de datos', 'Código del Municipio', 'Código del Departamento']
for col in enteros_con_nan:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype('Int64')

# 4. VERIFICACION FINAL DE LIMPIEZA:
print("--- VERIFICACION DE NUEVOS TIPOS DE DATOS ---")
print(df_clean[columnas_a_convertir + ['Código del Municipio']].dtypes)
print("\n" + "-"*50)

print("\n--- CONTEO DE NULOS REALES (POST-LIMPIEZA) ---")
print("Ahora si se evidencia la verdadera cantidad de datos faltantes:")
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])

                      LIMPIEZA DE DATOS                    

--- VERIFICACION DE NUEVOS TIPOS DE DATOS ---
Promedio                float64
Suma                    float64
No. de datos              Int64
Mediana                 float64
Percentil 98            float64
Máximo                  float64
Mínimo                  float64
Año                       Int64
ID Estacion               Int64
Código del Municipio      Int64
dtype: object

--------------------------------------------------

--- CONTEO DE NULOS REALES (POST-LIMPIEZA) ---
Ahora si se evidencia la verdadera cantidad de datos faltantes:
Representatividad Temporal    148
Código del Municipio            4
Tipo de Estación                2
dtype: int64


Para corregir las anomalías detectadas en la fase de diagnóstico, se aplicó un pipeline de transformación iterativo sobre las variables cuantitativas y geográficas:

- **Tratamiento de caracteres y espacios:**
 Se removieron los espacios en blanco espurios mediante desbroce de cadenas (strip) y se eliminaron los separadores de miles(,).

- **Conversión de tipos e imputación pasiva de nulos:**
 Se aplicó una conversión forzada a tipo numérico. Aquellos registros que contenían nulos camuflados (celdas vacías "" o marcadores de texto institucional) fueron coaccionados rigurosamente a valores NaN (Not a Number) nativos de Python.

- **Preservación de Estructura:** Las variables de identificación (como códigos geográficos, IDs y años) se formatearon bajo el tipo Int64 de Pandas para mitigar la conversión automática a flotantes debido a la presencia de valores faltantes.

**Estado del Dataset:** Las columnas ahora tienen propiedades matemáticas reales. Esto nos desvela el verdadero volumen de datos faltantes, permitiéndonos realizar un análisis exploratorio (EDA) estadísticamente riguroso sin sesgos por strings ocultos.

Para proseguir con la siguiente fase, es práctico eliminar estos registros con estas columnas en nulo, pues representan menos del 0.5% de los registros totales que hay (La Representatividad Temporal, justamente, solo un 0.49%) y no intervienen o generan un aporte significativo al resto del análisis y posterior modelo a entrenar.

In [33]:
df_clean = df_clean.dropna()

print("--- Dimensiones Actualizadas del Dataset ---")
print(f"Filas: {df_clean.shape[0]}, Columnas: {df_clean.shape[1]}")
print("\n")

--- Dimensiones Actualizadas del Dataset ---
Filas: 29531, Columnas: 28




## 5. Análisis Exploratorio de Datos

# **5.1 Análisis Univariado y Detección de Outliers**
Aquí se evalúa la distribución individual de los contaminantes principales. Con Diagramas de Caja (Boxplots), que son la herramienta perfecta para visualizar los cuartiles y exponer inmediatamente los outliers (todo punto que caiga fuera de los "bigotes" del gráfico). También histogramas para ver la forma de la curva (si está sesgada hacia valores de alta contaminación).

# **5.2 Análisis Bivariado**
Se cruzarán variables para encontrar patrones. Por ejemplo: ¿Cómo varía la concentración promedio dependiendo del Departamento? Esto ayuda a responder las preguntas geográficas planteadas en la Fase 1.

# **5.3 Análisis Multivariado**
Lo principal es una Matriz de Correlación. Esto es vital antes del Machine Learning, ya que si dos variables están perfectamente correlacionadas (ej. Máximo y Percentil 98), se podría eliminar una para evitar la multicolinealidad en el modelo.